In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import random
from tqdm import tqdm

# ------------------------------
# Configuration
# ------------------------------
MAX_X_LEN = 5
MAX_K_LEN = 3
MIN_X_LEN = 2
MIN_K_LEN = 2
BATCH_SIZE = 32
EMBED_DIM = 32
NUM_HEADS = 4
NUM_LAYERS = 2
LR = 1e-3
EPOCHS = 10
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ------------------------------
# Special tokens
# ------------------------------
PAD = 0
X_START = 1
X_END = 2
K_START = 3
K_END = 4
OP_CONV = 5
OP_TCONV = 6
VOCAB_SIZE = 20  # max integer in x/kernel + special tokens

# ------------------------------
# Synthetic Dataset
# ------------------------------
def conv1d(x, k):
    out_len = len(x) - len(k) + 1
    return [sum(x[i + j] * k[j] for j in range(len(k))) for i in range(out_len)]

def tconv1d(x, k):
    # transpose convolution (stride=1, no padding)
    out_len = len(x) + len(k) - 1
    y = [0] * out_len
    for i in range(len(x)):
        for j in range(len(k)):
            y[i + j] += x[i] * k[j]
    return y

class ConvDataset(Dataset):
    def __init__(self, num_samples=1000):
        self.data = []
        for _ in range(num_samples):
            x_len = random.randint(MIN_X_LEN, MAX_X_LEN)
            k_len = random.randint(MIN_K_LEN, MAX_K_LEN)
            x = [random.randint(7, VOCAB_SIZE-1) for _ in range(x_len)]
            k = [random.randint(1, VOCAB_SIZE-1) for _ in range(k_len)]
            op = random.choice([OP_CONV, OP_TCONV])
            y = conv1d(x, k) if op == OP_CONV else tconv1d(x, k)
            # input sequence: <X> x </X> <K> k </K> <OP>
            inp = [X_START] + x + [X_END] + [K_START] + k + [K_END] + [op]
            self.data.append((inp, y))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

# ------------------------------
# Collate function for variable length sequences
# ------------------------------
def collate_fn(batch):
    xs, ys = zip(*batch)
    max_len_x = max(len(x) for x in xs)
    max_len_y = max(len(y) for y in ys)
    
    x_pad = [x + [PAD]*(max_len_x - len(x)) for x in xs]
    y_pad = [y + [PAD]*(max_len_y - len(y)) for y in ys]
    
    x_tensor = torch.tensor(x_pad, dtype=torch.long)
    y_tensor = torch.tensor(y_pad, dtype=torch.float)  # output is numeric
    
    return x_tensor, y_tensor

# ------------------------------
# Transformer Decoder-Only Model
# ------------------------------
class TransformerICL(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(512, embed_dim)  # max sequence length
        decoder_layer = nn.TransformerDecoderLayer(embed_dim, num_heads)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers)
        self.fc_out = nn.Linear(embed_dim, 1)  # numeric output

    def forward(self, src, tgt):
        # src: [seq_len, batch]
        # tgt: [seq_len, batch]
        seq_len, batch_size = src.shape
        src_pos = torch.arange(seq_len, device=src.device).unsqueeze(1).expand(seq_len, batch_size)
        tgt_pos = torch.arange(tgt.shape[0], device=tgt.device).unsqueeze(1).expand(tgt.shape[0], batch_size)
        
        src_emb = self.embed(src) + self.pos_embed(src_pos)
        tgt_emb = self.embed(torch.clamp(tgt.long(), 0, VOCAB_SIZE-1)) + self.pos_embed(tgt_pos)
        
        memory = src_emb
        out = self.decoder(tgt_emb, memory)
        return self.fc_out(out).squeeze(-1)




In [3]:
# ------------------------------
# Training loop
# ------------------------------
dataset = ConvDataset(2000)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

model = TransformerICL(VOCAB_SIZE, EMBED_DIM, NUM_HEADS, NUM_LAYERS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()
track_loss = []
for epoch in range(EPOCHS):
    total_loss = 0
    tq = tqdm(loader)
    for x, y in tq:
        x, y = x.T.to(DEVICE), y.T.to(DEVICE)  # seq_len x batch
        optimizer.zero_grad()
        y_pred = model(x, y)
        loss = loss_fn(y_pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        track_loss.append(loss.item())
        tq.set_postfix(loss=sum(track_loss[-10:])/10)
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")


100%|████████████████████████████| 63/63 [00:00<00:00, 139.54it/s, loss=4.32e+4]


Epoch 1, Loss: 44554.7403


100%|████████████████████████████| 63/63 [00:00<00:00, 164.28it/s, loss=4.18e+4]


Epoch 2, Loss: 44261.8815


100%|████████████████████████████| 63/63 [00:00<00:00, 180.04it/s, loss=4.25e+4]


...

Epoch 8, Loss: 34166.9971


100%|████████████████████████████| 63/63 [00:00<00:00, 174.27it/s, loss=3.31e+4]


Epoch 9, Loss: 32225.8053


100%|████████████████████████████| 63/63 [00:00<00:00, 149.02it/s, loss=2.88e+4]

Epoch 10, Loss: 30085.9209


In [4]:
import torch
from torch.utils.data import DataLoader
import random

# Assuming dataset and model from previous code are already defined and trained

# ------------------------------
# Helper functions
# ------------------------------
def show_sample(x, y):
    # x is list of integers, y is list of numbers
    print("Input sequence:", x)
    print("Output sequence:", y)
    print("-" * 40)

def prepare_icl_input(samples):
    """
    Concatenate multiple (x, y) pairs for in-context learning.
    Each sample: (x_sequence, y_sequence)
    Returns: input tensor [seq_len, 1]
    """
    seq = []
    for x_seq, y_seq in samples:
        seq.extend(x_seq + [PAD])      # add padding between examples
        seq.extend([int(v) for v in y_seq])  # include y in context
        seq.append(PAD)
    return torch.tensor(seq, dtype=torch.long).unsqueeze(1)  # seq_len x 1

def infer_icl(model, context_samples, query_x, max_y_len=10):
    """
    context_samples: list of (x_seq, y_seq) for few-shot examples
    query_x: x sequence for which we want to predict y
    """
    model.eval()
    with torch.no_grad():
        # Build input: context + query
        inp_seq = []
        for x_seq, y_seq in context_samples:
            inp_seq.extend(x_seq + [PAD])
            inp_seq.extend([int(v) for v in y_seq])
            inp_seq.append(PAD)
        # append query
        inp_seq.extend(query_x + [PAD])
        src = torch.tensor(inp_seq, dtype=torch.long).unsqueeze(1).to(DEVICE)
        
        # autoregressive prediction
        pred_y = []
        tgt = torch.zeros((1,1), device=DEVICE)
        for _ in range(max_y_len):
            out = model(src, tgt)  # [seq_len, batch=1]
            next_val = out[-1,0].item()  # last timestep
            pred_y.append(next_val)
            next_tensor = torch.tensor([[next_val]], device=DEVICE)
            tgt = torch.cat([tgt, next_tensor], dim=0)
        return pred_y



In [5]:
# ------------------------------
# Display random samples
# ------------------------------
loader = DataLoader(dataset, batch_size=5, shuffle=True, collate_fn=collate_fn)
x_batch, y_batch = next(iter(loader))
x_batch = x_batch.T.tolist()
y_batch = y_batch.T.tolist()

print("=== Random samples ===")
for x_seq, y_seq in zip(x_batch, y_batch):
    show_sample(x_seq, y_seq)

# ------------------------------
# Zero-shot / few-shot ICL
# ------------------------------
# Pick few-shot context from dataset
context = [dataset[i] for i in range(3)]  # 3-shot
query = dataset[10][0]  # input sequence for query
true_y = dataset[10][1]

pred_y_0shot = infer_icl(model, [], query)
pred_y_3shot = infer_icl(model, context, query)

print("\n=== ICL Inference ===")
print("Query input:", query)
print("True output:", true_y)
print("0-shot prediction:", [round(v,2) for v in pred_y_0shot])
print("3-shot prediction:", [round(v,2) for v in pred_y_3shot])


=== Random samples ===
Input sequence: [1, 1, 1, 1, 1]
Output sequence: [70.0, 400.0, 456.0, 0.0, 180.0]
----------------------------------------
Input sequence: [7, 9, 19, 13, 15]
Output sequence: [177.0, 0.0, 466.0, 0.0, 177.0]
----------------------------------------
Input sequence: [10, 7, 11, 19, 11]
Output sequence: [323.0, 0.0, 373.0, 0.0, 297.0]
----------------------------------------
Input sequence: [15, 13, 17, 2, 7]
Output sequence: [255.0, 0.0, 237.0, 0.0, 357.0]
----------------------------------------
Input sequence: [2, 2, 8, 3, 17]
Output sequence: [135.0, 0.0, 0.0, 0.0, 135.0]
----------------------------------------
Input sequence: [3, 3, 7, 17, 2]
Output sequence: [0.0, 0.0, 0.0, 0.0, 204.0]
----------------------------------------


/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [26,0,0], thread: [0,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [26,0,0], thread: [1,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [26,0,0], thread: [2,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [26,0,0], thread: [3,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [26,0,0], thread: [4,0,0] Assertion 

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
